# Cleaning Dataset
## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow

## Converting January Data into Long Format

In [72]:
# load January data
jan = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-01.parquet')

# glimpse the structure of data
jan.shape

(4464, 799)

In [73]:
# analyzing the first 5 observations of first turbine
jan.iloc[:5, :17]

,utcTime,ALTA10-SCEWF-WTG150-AirDensity,ALTA10-SCEWF-WTG150-KPI-TemperatureAmbient-Merged,ALTA10-SCEWF-WTG150-KPI-RealPowerAC,ALTA10-SCEWF-WTG150-KPI-TheoreticalPower,ALTA10-SCEWF-WTG150-KPI-WindSpeed,ALTA10-SCEWF-WTG150-KPI-TemperatureAmbient,ALTA10-SCEWF-WTG150-RealPowerAC,ALTA10-SCEWF-WTG150-NacellePosition,ALTA10-SCEWF-WTG150-BladePosition2,ALTA10-SCEWF-WTG150-BladePosition3,ALTA10-SCEWF-WTG150-BladeSetpointPosition1,ALTA10-SCEWF-WTG150-BladeSetpointPosition2,ALTA10-SCEWF-WTG150-BladeSetpointPosition3,ALTA10-SCEWF-WTG150-WindSpeed,ALTA10-SCEWF-WTG150-WindDirection,ALTA10-SCEWF-WTG150-WindDirectionRelative
DataTimeStamp,,,,,,,,,,,,,,,,,
2026-01-01 00:00:00,2026-01-01 08:00:00,1.092081,8.299405,672.60510,746.68463,6.906036,8.299405,670.286714,118.785714,-1.056245,-0.643520,2.307242,-0.892182,-0.471435,6.907681,115.629870,-1.392857
2026-01-01 00:10:00,2026-01-01 08:10:00,0.800856,8.573889,541.30096,466.00030,6.609679,8.573889,542.747853,117.066667,-1.077751,-0.680390,2.354385,-0.845038,-0.424893,6.592723,107.211538,-10.056738
2026-01-01 00:20:00,2026-01-01 08:20:00,1.089478,8.972823,237.94884,292.86000,5.210308,8.972823,239.185173,118.812500,-0.445800,-0.039797,2.813232,-0.386191,0.032674,5.229490,122.834356,0.734694
2026-01-01 00:30:00,2026-01-01 08:30:00,0.800936,8.531316,503.22354,443.03992,6.510418,8.531316,504.556470,NaN,-1.205972,-0.786801,NaN,NaN,NaN,6.512685,121.493151,-6.506849
2026-01-01 00:40:00,2026-01-01 08:40:00,1.090836,8.621079,444.42767,500.71185,6.093416,8.621079,444.157211,130.000000,-1.205672,-0.786363,NaN,NaN,NaN,6.106378,121.117241,-8.119718


In [74]:
# creating index column
jan = jan.set_index('utcTime')

# separating the turbine and column name
jan_parts = jan.columns.str.extract(r"^ALTA\d+-SCEWF-(WTG\d+)-(.+)$")
jan_parts.columns = ['turbine', 'variable']

In [75]:
# analyzing if algorithm is messing up
bad = jan.columns[jan_parts['turbine'].isna()]
if len(bad):
    raise ValueError(f"Columns that didn't match the pattern: {list(bad)}")

In [76]:
# creating the columns
jan.columns = pd.MultiIndex.from_frame(jan_parts)

# creating the long version of the dataset
jan_long = jan.stack(level = 'turbine', future_stack = True).reset_index()
jan_long['turbine'] = long['turbine'].astype(str)
jan_long.columns.name = None
jan_long = long.sort_values(['turbine', 'utcTime']).reset_index(drop=True)

In [77]:
# analyzing shape of the dataset
jan_long.shape

(209808, 19)

In [78]:
# looking at first 5 values
jan_long.head()

,utcTime,turbine,AirDensity,KPI-TemperatureAmbient-Merged,KPI-RealPowerAC,KPI-TheoreticalPower,KPI-WindSpeed,KPI-TemperatureAmbient,RealPowerAC,NacellePosition,BladePosition2,BladePosition3,BladeSetpointPosition1,BladeSetpointPosition2,BladeSetpointPosition3,WindSpeed,WindDirection,WindDirectionRelative,BladePosition1
0,2026-01-01 08:00:00,WTG150,1.092081,8.299405,672.60510,746.68463,6.906036,8.299405,670.286714,118.785714,-1.056245,-0.643520,2.307242,-0.892182,-0.471435,6.907681,115.629870,-1.392857,NaN
1,2026-01-01 08:10:00,WTG150,0.800856,8.573889,541.30096,466.00030,6.609679,8.573889,542.747853,117.066667,-1.077751,-0.680390,2.354385,-0.845038,-0.424893,6.592723,107.211538,-10.056738,NaN
2,2026-01-01 08:20:00,WTG150,1.089478,8.972823,237.94884,292.86000,5.210308,8.972823,239.185173,118.812500,-0.445800,-0.039797,2.813232,-0.386191,0.032674,5.229490,122.834356,0.734694,NaN
3,2026-01-01 08:30:00,WTG150,0.800936,8.531316,503.22354,443.03992,6.510418,8.531316,504.556470,NaN,-1.205972,-0.786801,NaN,NaN,NaN,6.512685,121.493151,-6.506849,NaN
4,2026-01-01 08:40:00,WTG150,1.090836,8.621079,444.42767,500.71185,6.093416,8.621079,444.157211,130.000000,-1.205672,-0.786363,NaN,NaN,NaN,6.106378,121.117241,-8.119718,NaN


In [102]:
mapping = {'utcTime': 'utc', 'turbine': 'turbine', 'AirDensity': 'aid_density', 'KPI-TemperatureAmbient-Merged': 'KPI_temp_ambient_merged', 
           'KPI-TemperatureAmbient': 'KPI_temp_ambient', 'KPI-TheoreticalPower': 'KPI_theoretical_power', 'KPI-RealPowerAC': 'KPI_real_power_ac',
           'RealPowerAC': 'real_power_ac', 'KPI-WindSpeed': 'KPI_wind_speed', 'WindSpeed': 'wind_speed', 'WindDirection': 'wind_direction', 
           'WindDirectionRelative': 'wind_direction_relative', 'NacellePosition': 'nacelle_pos', 'BladePosition1': 'blade_1_pos', 
           'BladePosition2': 'blade_2_pos', 'BladePosition3': 'blade_3_pos', 'BladeSetpointPosition1': 'blade_1_setpos', 
           'BladeSetpointPosition2': 'blade_2_setpos', 'BladeSetpointPosition3': 'blade_3_setpoint'}

# checking to make sure all turbines are present
out = jan_long[list(mapping)].rename(columns=mapping)

out.head()

,utc,turbine,aid_density,KPI_temp_ambient_merged,KPI_temp_ambient,KPI_theoretical_power,KPI_real_power_ac,real_power_ac,KPI_wind_speed,wind_speed,wind_direction,wind_direction_relative,nacelle_pos,blade_1_pos,blade_2_pos,blade_3_pos,blade_1_setpos,blade_2_setpos,blade_3_setpoint
0,2026-01-01 08:00:00,WTG150,1.092081,8.299405,8.299405,746.68463,672.60510,670.286714,6.906036,6.907681,115.629870,-1.392857,118.785714,NaN,-1.056245,-0.643520,2.307242,-0.892182,-0.471435
1,2026-01-01 08:10:00,WTG150,0.800856,8.573889,8.573889,466.00030,541.30096,542.747853,6.609679,6.592723,107.211538,-10.056738,117.066667,NaN,-1.077751,-0.680390,2.354385,-0.845038,-0.424893
2,2026-01-01 08:20:00,WTG150,1.089478,8.972823,8.972823,292.86000,237.94884,239.185173,5.210308,5.229490,122.834356,0.734694,118.812500,NaN,-0.445800,-0.039797,2.813232,-0.386191,0.032674
3,2026-01-01 08:30:00,WTG150,0.800936,8.531316,8.531316,443.03992,503.22354,504.556470,6.510418,6.512685,121.493151,-6.506849,NaN,NaN,-1.205972,-0.786801,NaN,NaN,NaN
4,2026-01-01 08:40:00,WTG150,1.090836,8.621079,8.621079,500.71185,444.42767,444.157211,6.093416,6.106378,121.117241,-8.119718,130.000000,NaN,-1.205672,-0.786363,NaN,NaN,NaN


## Converting Remaining Months into Long Format

In [89]:
# Load raw data
jan_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-02.parquet')
feb_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-02.parquet')
mar_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-03.parquet')
apr_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-04.parquet')
may_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-05.parquet')
jun_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-06.parquet')
jul_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-07.parquet')
aug_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-08.parquet')
sep_wide = pd.read_parquet('../data/raw/Alta10_Turbine_Points_10m_2026-09.parquet')

In [103]:
# reordering and renaming variables with a dictionary
mapping = {'utcTime': 'utc', 'turbine': 'turbine', 'AirDensity': 'aid_density', 'KPI-TemperatureAmbient-Merged': 'KPI_temp_ambient_merged', 
           'KPI-TemperatureAmbient': 'KPI_temp_ambient', 'KPI-TheoreticalPower': 'KPI_theoretical_power', 'KPI-RealPowerAC': 'KPI_real_power_ac',
           'RealPowerAC': 'real_power_ac', 'KPI-WindSpeed': 'KPI_wind_speed', 'WindSpeed': 'wind_speed', 'WindDirection': 'wind_direction', 
           'WindDirectionRelative': 'wind_direction_relative', 'NacellePosition': 'nacelle_pos', 'BladePosition1': 'blade_1_pos', 
           'BladePosition2': 'blade_2_pos', 'BladePosition3': 'blade_3_pos', 'BladeSetpointPosition1': 'blade_1_setpos', 
           'BladeSetpointPosition2': 'blade_2_setpos', 'BladeSetpointPosition3': 'blade_3_setpoint'}

# defining a function to convert the wide format datasets into long format datasets
def reshape_year(df):
    # defines utcTime as index for rows in new dataset
    df = df.set_index('utcTime')

    # splits the columns into the turbine name and variable name
    parts = df.columns.str.extract(r"^ALTA\d+-SCEWF-(WTG\d+)-(.+)$")
    parts.columns = ['turbine', 'variable']

    # outputs problematic columns from this split
    bad = df.columns[parts['turbine'].isna()]
    if len(bad):
        raise ValueError(f"Columns that didn't match the pattern: {list(bad)}")

    # creates a copy of the dataframe
    df = df.copy()
    df.columns = pd.MultiIndex.from_frame(parts)

    # creates the long version of the dataset
    long = df.stack(level = 'turbine', future_stack = True).reset_index()
    long['turbine'] = long['turbine'].astype(str)
    long.columns.name = None
    long.sort_values(['turbine', 'utcTime']).reset_index(drop=True)

    # returns new dataset with renamed and reordered columns
    return long[list(mapping)].rename(columns=mapping)

In [104]:
# applying the reshape_year() function to each monthly dataset
jan = reshape_year(jan_wide)
feb = reshape_year(feb_wide)
mar = reshape_year(mar_wide)
apr = reshape_year(apr_wide)
may = reshape_year(may_wide)
jun = reshape_year(jun_wide)
jul = reshape_year(jul_wide)
aug = reshape_year(aug_wide)
sep = reshape_year(sep_wide)

In [114]:
# defining a list of the monthly datasets
monthly = [jan, feb, mar, apr, may, jun, jul, aug, sep]

# finding the number of observations in each dataset
for m in monthly:
    print(m.shape[1])

798
785
785
771
771
771
771
771
771


In [105]:
feb.shape

(189504, 19)

In [106]:
mar.shape

(209526, 19)

In [107]:
apr.shape

(203040, 19)

In [108]:
may.shape

(209808, 19)

In [109]:
jun.shape

(203040, 19)

In [110]:
jul.shape

(209808, 19)

In [111]:
aug.shape

(209808, 19)

In [112]:
sep.shape

(189504, 19)

In [113]:
feb.dtypes

utc                        datetime64[ns]
turbine                               str
aid_density                       float64
KPI_temp_ambient_merged           float64
KPI_temp_ambient                  float64
KPI_theoretical_power             float64
KPI_real_power_ac                 float64
real_power_ac                     float64
KPI_wind_speed                    float64
wind_speed                        float64
wind_direction                    float64
wind_direction_relative           float64
nacelle_pos                       float64
blade_1_pos                       float64
blade_2_pos                       float64
blade_3_pos                       float64
blade_1_setpos                    float64
blade_2_setpos                    float64
blade_3_setpoint                  float64
dtype: object